# OntoKG-EQ — multi-model oracle faithfulness panel (free GPU)

Runs the provenance-grounded **oracle harness** over a **panel of open models** so the "complementary failure modes" finding becomes a *powered* result rather than n=2.

**Before running:** Settings → Accelerator = **GPU T4 x2**, Internet = **On**, and **Add Input** the dataset with the four `*.ttl` files (`demo_psx_inferred.ttl`, `demo_msx_inferred.ttl`, `demo_idx_inferred.ttl`, `demo_idx_scaled.ttl`).

Cell 1 installs deps; Cell 2 defines the harness (case builders, scorer, Wilson CI, McNemar, a robust 4-bit loader); Cell 3 loops the model panel, clears GPU memory between models, and saves:
`panel_results.{md,csv}` (per-model table with 95% CIs), `panel_mcnemar.csv` (pairwise significance on scaled provenance), `panel_percase.csv` (per-case flags). Paste `panel_results.md` back and I'll drop it into Table 7. ~30–60 min for 8 small/mid models on a T4.



In [ ]:
# Pinned environment (exact resolved versions + model revisions are recorded to environment_lock.json below)
!pip -q install rdflib==7.6.0 transformers==4.44.2 accelerate==0.33.0 bitsandbytes==0.43.3 sentencepiece==0.2.0


In [ ]:
import os, re, glob, gc, csv, math, itertools
from rdflib import Graph, RDF, URIRef
from rdflib.namespace import Namespace
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
CORE = Namespace("https://w3id.org/ontokg-eq#")
OUT  = "/kaggle/working"

def _find(name):
    h = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    return h[0] if h else None
BASE_DIR   = os.path.dirname(_find("demo_idx_inferred.ttl") or "/kaggle/input/x")
SCALED_TTL = _find("demo_idx_scaled.ttl")
print("base dir:", BASE_DIR, "| scaled ttl:", SCALED_TTL)

def short(x): return str(x).split("#")[-1] if x else ""
PRED = re.compile(r"\b(because|due to|driven by|will|likely|expected to|forecast|predict|target|recommend|buy|sell|cause|outlook|going to|should)\b", re.I)
NUM  = re.compile(r"-?\d+\.\d+")

def build_cases():
    cases=[]
    for m in ["psx","msx","idx"]:
        fp=os.path.join(BASE_DIR,f"demo_{m}_inferred.ttl")
        if not os.path.exists(fp): continue
        g=Graph().parse(fp,format="turtle")
        for f in g.subjects(RDF.type, CORE.AnalyticalFinding):
            ft=str(next(g.objects(f,CORE.hasFindingType),""))
            if ft not in ("relative-outperformer","fundamentals-market-divergence"): continue
            ent=next(g.objects(f,CORE.concernsEntity),None)
            label=str(next(g.objects(ent,CORE.hasCompanyName),short(ent)))
            b=URIRef(str(f)+"_bundle"); atoms=[]; nums=[]
            for o in g.objects(b,CORE.includesObservation):
                mn=str(next(g.objects(o,CORE.hasMetricName),"")); v=next(g.objects(o,CORE.hasMetricValue),None)
                tgt=short(next(g.objects(o,CORE.isObservationOf),None))
                if v is not None:
                    fv=round(float(v),2); atoms.append(f"{tgt} {mn} = {fv}"); nums.append(fv)
            ev=next(g.objects(b,CORE.containsEvidenceItem),None)
            date=str(next(g.objects(ev,CORE.hasAnnouncementDate),"")) if ev else ""
            src=next(g.objects(ev,CORE.hasEvidenceSource),None) if ev else None
            stype=str(next(g.objects(src,CORE.hasSourceType),"")) if src else ""
            ctx="Facts present in the validated knowledge graph:\n - "+"\n - ".join(atoms)
            if date: ctx+=f"\n - evidence: {label} FY results announced {date}"
            if src:  ctx+=f"\n - source: {short(src)} ({stype})"
            q=(f"Did {label} outperform both its sector and the broad-market benchmark over the post-report window? Explain, citing the evidence."
               if ft=="relative-outperformer" else
               f"Did {label} show stronger fundamentals but a weaker market response than the benchmark? Explain, citing the evidence.")
            onto=f"{label}: "+"; ".join(atoms)+". "+(f"Evidence: FY results announced {date} (source: {short(src)})." if src else "")
            cases.append(dict(market=m,label=label,ftype=ft,question=q,context=ctx,
                              factnums=set(round(n,2) for n in nums),
                              prov_tokens=[date, short(src) if src else ""],   # specific source/date only
                              onto_answer=onto))
    return cases

def build_scaled_cases(ttl):
    if not ttl or not os.path.exists(ttl): return []
    g=Graph().parse(ttl,format="turtle")
    post={}; bench={}; sect={}; growth={}
    for o in g.subjects(CORE.hasMetricName,None):
        mn=str(next(g.objects(o,CORE.hasMetricName))); v=next(g.objects(o,CORE.hasMetricValue),None)
        if v is None: continue
        val=float(v); ent=next(g.objects(o,CORE.isObservationOf),None); w=next(g.objects(o,CORE.observedOverWindow),None)
        if   mn=="post-report window return %" and w is not None: post[w]=(ent,val)
        elif mn=="benchmark window return %"  and w is not None: bench[w]=val
        elif mn=="sector window return %"     and w is not None: sect[w]=val
        elif mn=="YoY profit growth %": growth[ent]=val
    ann={}
    for a in g.subjects(RDF.type, CORE.Announcement):
        e=next(g.objects(a,CORE.aboutCompany),None); d=str(next(g.objects(a,CORE.hasAnnouncementDate),""))
        s=next(g.objects(a,CORE.hasEvidenceSource),None)
        ann[e]=(d, short(s) if s else "", str(next(g.objects(s,CORE.hasSourceType),"")) if s else "")
    cases=[]
    for w,(ent,cret) in post.items():
        br=bench.get(w); sr=sect.get(w); gr=growth.get(ent)
        label=str(next(g.objects(ent,CORE.hasCompanyName),short(ent)))
        d,src,st=ann.get(ent,("","",""))
        prov=[d,src]   # specific source/date only
        if sr is not None and br is not None and cret>sr and cret>br:
            atoms=[f"{label} return = {round(cret,2)}", f"sector return = {round(sr,2)}", f"benchmark return = {round(br,2)}"]
            cases.append(dict(market="idx_scaled",label=label,ftype="relative-outperformer",
                question=f"Did {label} outperform both its sector and the broad-market benchmark over the window? Explain, citing the source.",
                context="Facts present in the validated knowledge graph:\n - "+"\n - ".join(atoms)+(f"\n - evidence: FY results announced {d} (source: {src})" if d else ""),
                factnums={round(cret,2),round(sr,2),round(br,2)}, prov_tokens=prov,
                onto_answer=f"{label}: "+"; ".join(atoms)+f". Evidence: FY results announced {d} (source: {src})."))
        if gr is not None and br is not None and gr>0 and cret<br:
            atoms=[f"{label} YoY profit growth = {round(gr,2)}", f"{label} return = {round(cret,2)}", f"benchmark return = {round(br,2)}"]
            cases.append(dict(market="idx_scaled",label=label,ftype="fundamentals-market-divergence",
                question=f"Did {label} report stronger fundamentals but a weaker market response than the benchmark? Explain, citing the source.",
                context="Facts present in the validated knowledge graph:\n - "+"\n - ".join(atoms)+(f"\n - evidence: FY results announced {d} (source: {src})" if d else ""),
                factnums={round(gr,2),round(cret,2),round(br,2)}, prov_tokens=prov,
                onto_answer=f"{label}: "+"; ".join(atoms)+f". Evidence: FY results announced {d} (source: {src})."))
    return cases

def score(ans,factnums,prov,tol=0.05):
    ns=[round(float(x),2) for x in NUM.findall(ans)]
    mt=[n for n in ns if any(abs(n-t)<=tol for t in factnums)]
    rc=[t for t in factnums if any(abs(n-t)<=tol for n in ns)]
    return {"num_faithful":round(len(mt)/len(ns),2) if ns else 1.0,     # PRECISION
            "num_recall":round(len(rc)/len(factnums),2) if factnums else 1.0,  # RECALL/completeness
            "halluc":len(ns)-len(mt),
            "unsupported":len(PRED.findall(ans)),
            "provenance":1 if any(t and str(t).lower() in ans.lower() for t in prov) else 0}

def wilson(k,n,z=1.96):
    if n==0: return (0.0,0.0)
    p=k/n; d=1+z*z/n; c=(p+z*z/(2*n))/d; h=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))/d
    return (max(0.0,c-h), min(1.0,c+h))

def mcnemar_exact(a,b):
    b01=sum(1 for x,y in zip(a,b) if x==0 and y==1)
    c10=sum(1 for x,y in zip(a,b) if x==1 and y==0)
    n=b01+c10
    if n==0: return 1.0,b01,c10
    from math import comb
    k=min(b01,c10)
    p=min(1.0, 2*sum(comb(n,i) for i in range(0,k+1))/(2**n))
    return p,b01,c10

def load_model(name, hf_token=None):
    tok=AutoTokenizer.from_pretrained(name, token=hf_token)
    cfg=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                           bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
    last=None
    for dm in [{"":0},{"":1},"auto"]:
        try:
            gc.collect(); torch.cuda.empty_cache()
            mm={0:"13GiB",1:"13GiB","cpu":"40GiB"} if dm=="auto" else None
            mdl=AutoModelForCausalLM.from_pretrained(name, quantization_config=cfg, device_map=dm,
                    max_memory=mm, low_cpu_mem_usage=True, token=hf_token)
            return tok, mdl
        except Exception as e:
            last=e
    raise RuntimeError(f"4-bit load failed for {name}: {repr(last)[:160]}")

def llm_answer(pipe,q,ctx):
    prompt=(f"You are a financial analysis assistant. Answer ONLY using the facts in the context; "
            f"cite the official source. Be concise (2-3 sentences).\n\nQuestion: {q}\n\nContext:\n{ctx}\n\nAnswer:")
    out=pipe([{"role":"user","content":prompt}], max_new_tokens=200, do_sample=False)
    gen=out[0]["generated_text"]
    return gen[-1]["content"] if isinstance(gen,list) else str(gen)



In [ ]:
# ===== MODEL PANEL (all ungated, fit a single T4 in 4-bit). Trim/extend as you like. =====
MODELS = [
    "Qwen/Qwen2.5-0.5B-Instruct",
    "Qwen/Qwen2.5-1.5B-Instruct",
    "Qwen/Qwen2.5-3B-Instruct",
    "Qwen/Qwen2.5-7B-Instruct",
    "HuggingFaceTB/SmolLM2-1.7B-Instruct",
    "microsoft/Phi-3.5-mini-instruct",
    "mistralai/Mistral-7B-Instruct-v0.3",   # if this errors 'gated', remove it or add an HF token
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
]
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")

base   = build_cases()
scaled = build_scaled_cases(SCALED_TTL)
print(f"cases: base={len(base)}  scaled={len(scaled)} "
      f"({sum(c['ftype']=='relative-outperformer' for c in scaled)} CQ3 + "
      f"{sum(c['ftype']=='fundamentals-market-divergence' for c in scaled)} CQ1)")

onto = [score(c["onto_answer"], c["factnums"], c["prov_tokens"]) for c in base+scaled]

def agg(ss, name):
    n=len(ss)
    if n==0: return [name,0,1.00,1.00,0.00,0.00,1.00,"—"]
    prov=sum(s["provenance"] for s in ss); lo,hi=wilson(prov,n)
    return [name,n,round(sum(s["num_faithful"] for s in ss)/n,3),
            round(sum(s["num_recall"] for s in ss)/n,3),
            round(sum(s["halluc"] for s in ss)/n,2),
            round(sum(s["unsupported"] for s in ss)/n,2),
            round(prov/n,2), f"[{lo:.2f}, {hi:.2f}]"]

rows=[agg(onto,"OntoKG-EQ / deterministic linearizer (design guarantee)")]
percase=[]; scaled_prov={}; revisions={}
for m in MODELS:
    print("="*60, "\nLoading", m)
    try:
        tok, mdl = load_model(m, HF_TOKEN); pipe = pipeline("text-generation", model=mdl, tokenizer=tok)
        revisions[m]=getattr(mdl.config,"_commit_hash",None)
    except Exception as e:
        print("SKIP", m, ":", repr(e)[:160]); continue
    sb=[]; ss=[]; flags=[]
    for coh, cases, store in [("curated", base, sb), ("scaled-64", scaled, ss)]:
        for i,c in enumerate(cases,1):
            a=llm_answer(pipe, c["question"], c["context"]); s=score(a, c["factnums"], c["prov_tokens"]); store.append(s)
            percase.append({"model":m,"cohort":coh,"label":c["label"],"cq":c["ftype"],
                            "provenance":s["provenance"],"num_faithful":s["num_faithful"],"num_recall":s["num_recall"],"halluc":s["halluc"]})
            if coh=="scaled-64": flags.append(s["provenance"])
    scaled_prov[m]=flags
    rows.append(agg(sb, f"{m.split('/')[-1]} (curated)"))
    rows.append(agg(ss, f"{m.split('/')[-1]} (scaled-64)"))
    print(f"  {m.split('/')[-1]}: scaled provenance={sum(flags)}/{len(flags)}")
    del pipe, mdl, tok; gc.collect(); torch.cuda.empty_cache()

# ---- main table ----
hdr=["method","n","numeric_precision","numeric_recall","hallucinated_numbers","unsupported","provenance","provenance_95CI"]
with open(f"{OUT}/panel_results.csv","w",newline="") as f:
    w=csv.writer(f); w.writerow(hdr); [w.writerow(r) for r in rows]
md="# Multi-model oracle faithfulness panel\n\n"
md+="| Method | n | num. precision | num. recall | halluc. | unsupported | provenance | 95% CI |\n|---|--:|--:|--:|--:|--:|--:|---|\n"
for r in rows: md+="| "+" | ".join(str(x) for x in r)+" |\n"
open(f"{OUT}/panel_results.md","w").write(md); print("\n"+md)

# ---- pairwise McNemar on scaled provenance (same 41 cases) ----
mods=[m for m in MODELS if len(scaled_prov.get(m,[]))==len(scaled) and len(scaled)>0]
mm="model_A,model_B,only_B_cites,only_A_cites,p_exact\n"
print("\nPairwise McNemar (scaled provenance, exact two-sided p):")
for a,b in itertools.combinations(mods,2):
    p,b01,c10=mcnemar_exact(scaled_prov[a],scaled_prov[b])
    mm+=f"{a},{b},{b01},{c10},{p:.4g}\n"
    print(f"  {a.split('/')[-1]:28s} vs {b.split('/')[-1]:28s} p={p:.4g}")
open(f"{OUT}/panel_mcnemar.csv","w").write(mm)

with open(f"{OUT}/panel_percase.csv","w",newline="") as f:
    w=csv.DictWriter(f, fieldnames=["model","cohort","label","cq","provenance","num_faithful","num_recall","halluc"])
    w.writeheader(); [w.writerow(r) for r in percase]
# ---- environment lock: exact resolved package versions + model revisions ----
import sys, platform
try:
    import transformers, accelerate, bitsandbytes
    pkg={"python":sys.version.split()[0],"platform":platform.platform(),"torch":torch.__version__,
         "transformers":transformers.__version__,"accelerate":accelerate.__version__,
         "bitsandbytes":getattr(bitsandbytes,"__version__","?"),
         "cuda":getattr(torch.version,"cuda",None),
         "gpu":(torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)}
except Exception as e:
    pkg={"error":str(e)}
import json as _json
_json.dump({"packages":pkg,"model_revisions":revisions}, open(f"{OUT}/environment_lock.json","w"), indent=2)
print("wrote environment_lock.json:", pkg)
print("\nSaved to /kaggle/working: panel_results.{md,csv}, panel_mcnemar.csv, panel_percase.csv, environment_lock.json")

